In [1]:
!pip install ollama -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.11.0
    Uninstalling typing_extensions-4.11.0:
      Successfully uninstalled typing_extensions-4.11.0
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.20.1
    Uninstalling pydantic_core-2.20.1:
      Successfully uninstalled pydantic_core-2.20.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.8.2
    Uninstalling pydantic-2.8.2:
      Successfully uninstalled pydantic-2.8.2


In [2]:
import ollama

response = ollama.chat(
    model="deepseek-r1",
    messages=[
        {"role": "user", "content": "Explain Newton's second law of motion"},
    ],
)
print(response["message"]["content"])

<think>
Okay, so I need to explain Newton's second law of motion. Hmm, I remember learning a bit about this in physics class, but I'm not totally confident about all the details. Let me try to break it down step by step.

First off, Newton's laws are three fundamental principles that describe how objects move and interact. The first one was probably about inertia, which means an object at rest stays at rest unless acted upon by a force. But now I'm focusing on the second law.

I think the second law has something to do with acceleration, mass, and force. Something like F equals ma or maybe it's m times a? Wait, wasn't it force equals mass times acceleration? Yeah, that sounds familiar: F = ma. So, if you have an object, its acceleration depends on two things: the net force acting on it and its mass.

Let me think about what each part means. The net force is all the forces acting on the object added up together vectorially, right? So if multiple forces are pushing or pulling on an objec

In [3]:
pip install gradio -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install langchain -q

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install -U langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 4.5 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install chromadb -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.3 which is incompatible.
tensorflow-metadata 1.15.0 requires protobuf<5,>=4.25.2; python_version >= "3.11", but you have protobuf 5.29.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [15]:
pip install pymupdf -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gradio as gr
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
import ollama
import json
import re

def process_pdf(pdf_bytes):
    """
    Processes uploaded PDF and returns necessary data structures.

    Args:
        pdf_bytes (bytes): Bytes of the uploaded PDF file.

    Returns:
        tuple: Tuple containing text splitter, vectorstore, and retriever objects.
    """
    if pdf_bytes is None:
        return None, None, None

    loader = PyMuPDFLoader(pdf_bytes)
    data = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = text_splitter.split_documents(data)

    embeddings = OllamaEmbeddings(model="deepseek-r1")
    vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)
    retriever = vectorstore.as_retriever()

    return text_splitter, vectorstore, retriever

def combine_docs(docs):
    """
    Combines page content of retrieved documents into a single string.

    Args:
    docs (list): List of retrieved documents.

    Returns:
    str: Combined content of retrieved documents.
    """
    return "\n\n".join(doc.page_content for doc in docs)

def ollama_llm(question, context):
    """
    Sends a prompt to Ollama for question answering.

    Args:
        question (str): User's question about the PDF.
        context (str): Combined content of retrieved documents.

    Returns:
        str: The final answer with the thinking part removed.
    """
    formatted_prompt = f"Question: {question}\n\nContext: {context}"
    response = ollama.chat(model="deepseek-r1", messages=[{'role': 'user', 'content': formatted_prompt}])
    response_content = response['message']['content']
    
    # Remove content between <think> and </think> tags
    final_answer = re.sub(r'<think>.*?</think>', '', response_content, flags=re.DOTALL).strip()
    return final_answer

def rag_chain(question, text_splitter, vectorstore, retriever):
    """
    Retrieves relevant documents from the vector store and uses Ollama for answering.

    Args:
        question (str): User's question about the PDF.
        text_splitter (object): Text splitter object.
        vectorstore (object): Chroma vector store object.
        retriever (object): Retriever object from the vector store.

    Returns:
        str: The final answer with the thinking part removed.
    """
    retrieved_docs = retriever.invoke(question)
    formatted_content = combine_docs(retrieved_docs)
    return ollama_llm(question, formatted_content)

def ask_question(pdf_bytes, question):
    """
    Asks a question about the PDF and retrieves an answer using RAG.

    Args:
        pdf_bytes (bytes): Bytes of the uploaded PDF file (can be None).
        question (str): User's question about the PDF.

    Returns:
        str: The final answer with the thinking part removed if PDF is uploaded, otherwise None.
    """
    text_splitter, vectorstore, retriever = process_pdf(pdf_bytes)
    if text_splitter is None:
        return None  # No PDF uploaded

    result = rag_chain(question, text_splitter, vectorstore, retriever)
    return result

interface = gr.Interface(
    fn=ask_question,
    inputs=[gr.File(label="Upload PDF (optional)"), gr.Textbox(label="Ask a question")],
    outputs="text",
    title="Ask questions about your PDF",
    description="Use DeepSeek-R1 to answer your questions about the uploaded PDF document. The thinking part is removed from the response.",
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
